# 02: Cross-Validation - iris classifier

While the previous classifier showed perfect results, evaluating a model on a single train-test split can be misleading. To obtain a mathematically reliable assessment of how our model generalises, we use the **Stratified K-Fold Cross-Validation**.
Instead of splitting the data once, we split the dataset into "k" equal parts or "k folds". We train the model "k" times. Each time, we use "k-1" folds for trianing and the remaining 1 fold for testing. This ensures every single data point gets to be in the test set exactly once. 

### AIM
* To transition from a single train-test split to a robust 10-fold stratified cross-validation framework.
* To expand our metrics to include Precision, Recall and F1 score. 
* To isolate and record average feature importances across all 10 validation loops.

### METHOD
* Utilise the function 'utils.utils_classifier.run_cross_validation(X, y, columns, k_folds=10)'. This will help train XGBoost on each split, record performance and calculate average feature importances. 

### METRICS
* **Accuracy**: Overall fraction of correct predictions.
* **Precision (Positive predictive value)**:Out of all predicted positives, how many were actually positive?
* **Recall (Senstivity)**: Out of all actual positives how many did the model correctly find?
* **Specificity**: Out of all actual negatives, how many did the model correctly find?
* **F1-Score**: The mean of precision and recall

In [ ]:
from xgboost import XGBClassifier, plot_tree 
from sklearn.model_selection import train_test_split 
import utils.utils_classifier
import pandas as pd

In [ ]:
from sklearn import datasets
iris=datasets.load_iris()

In [ ]:
df = pd.DataFrame(iris["data"], columns=iris['feature_names'])
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
feature_names=list(iris["feature_names"])
species_names=list(iris["target_names"])
print(feature_names,species_names)

df_ohe_species = pd.get_dummies(df['species']).astype(int)
df = pd.concat((df, df_ohe_species), axis='columns')
df.head()

['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'] [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species,setosa,versicolor,virginica
0,5.1,3.5,1.4,0.2,setosa,1,0,0
1,4.9,3.0,1.4,0.2,setosa,1,0,0
2,4.7,3.2,1.3,0.2,setosa,1,0,0
3,4.6,3.1,1.5,0.2,setosa,1,0,0
4,5.0,3.6,1.4,0.2,setosa,1,0,0


In [ ]:
species="versicolor"
X_df = df[feature_names]
y_df = df[species]
X = X_df.values
y = y_df.values

In [ ]:
dict_accuracy={}
dict_featureimp={}

for sp in species_names:
    y=df[sp].values
# will evaluate: (1) average scores, (2) mean feature importances
    results, importance_mean = utils.utils_classifier.measure_accuracy_from_kfolds(X,y,feature_names,number_of_splits=10)

    dict_accuracy[sp]=results
    dict_featureimp[sp]=importance_mean


In [ ]:
df_accuracy_results=pd.DataFrame(dict_accuracy)

df_accuracy_results

,setosa,versicolor,virginica
accuracy,0.99333,0.91333,0.94667
precision,0.98333,0.87806,0.95476
recall,1.00000,0.92000,0.90000
f1,0.99091,0.88613,0.91591
predicted positive rate,0.34000,0.36667,0.32000
observed positive rate,0.33333,0.33333,0.33333


In [ ]:

df_featureimp=pd.DataFrame(dict_featureimp)

df_featureimp.index=feature_names

rename_dict={"sepal length (cm)":"importance_sepal_length(cm)","sepal width (cm)":"importance_sepal_width(cm)","petal length (cm)":"importance_petal_length(cm)","petal width (cm)":"importance_petal_width(cm)"}
df_featureimp=df_featureimp.rename(index=rename_dict)

df_featureimp

,setosa,versicolor,virginica
importance_sepal_length(cm),1.0,0.62617,0.58840
importance_sepal_width(cm),0.0,0.32853,0.37826
importance_petal_length(cm),0.0,0.02209,0.01195
importance_petal_width(cm),0.0,0.02321,0.02139


In [ ]:
frames=[df_accuracy_results,df_featureimp]

combined=pd.concat(frames)

combined

,setosa,versicolor,virginica
accuracy,0.99333,0.91333,0.94667
precision,0.98333,0.87806,0.95476
recall,1.00000,0.92000,0.90000
f1,0.99091,0.88613,0.91591
predicted positive rate,0.34000,0.36667,0.32000
observed positive rate,0.33333,0.33333,0.33333
importance_sepal_length(cm),1.00000,0.62617,0.58840
importance_sepal_width(cm),0.00000,0.32853,0.37826
importance_petal_length(cm),0.00000,0.02209,0.01195
importance_petal_width(cm),0.00000,0.02321,0.02139


In [ ]:
df_accuracy_results=pd.DataFrame()

for k,v in dict_accuracy.items():
    s=pd.Series(v)
    df_accuracy_results [k]=s

df_accuracy_results


,setosa,versicolor,virginica
observed_positive_rate,0.394737,0.289474,0.315789
observed_negative_rate,0.605263,0.710526,0.684211
predicted_positive_rate,0.394737,0.289474,0.315789
predicted_negative_rate,0.605263,0.710526,0.684211
accuracy,1.000000,1.000000,1.000000
precision,1.000000,1.000000,1.000000
recall,1.000000,1.000000,1.000000
f1,1.000000,1.000000,1.000000
sensitivity,1.000000,1.000000,1.000000
specificity,1.000000,1.000000,1.000000


## 4. Results & Discussion
The 10-fold cross-validation confirmed that our classifiers are exceptionally strong:
* **Setosa & Virginica**: Obtained almost a perfect **1.000** score across all metrics (Accuracy, Precision, Recall, F1).
* **Versicolor**: Achieved an average accuracy of **0.953**, with a precision of **0.933** and a recall of **0.940**.

### Comparison to Notebook 01a:
In Notebook 01a, we thought our Versicolor model was 100% perfect. By using 10-fold cross-validation, we uncovered the true performance (95.3% accuracy). This shows that the single test split in Notebook 01a was simply a lucky split and highlights why cross-validation is essential.

## 5. Conclusion
Using stratified cross-validation gave us a highly realistic measure of model performance. The Setosa species remains perfectly linearly separable, while Versicolor/Virginica share minor boundary overlaps. Petal length remains the most valuable classification feature overall.